# fMRI Beta Regression Analysis
**Data Mining Class Project**  
Predicting 8 brain-region beta values from 21 questionnaire variables using:
Multiple Linear Regression, Ridge, and Lasso (with 5-fold CV, 10-fold CV, and 80/20 split).

### How to run
1. Run **Cell 1** — you will be prompted to upload `BetasCorrelation.xlsx`.
2. Then choose **Runtime → Run all** for the rest.
3. All CSVs and PNG plots will appear in the **Files** panel (left sidebar).


In [ ]:
# All required libraries are pre-installed in Colab.
# Uncomment if anything is missing:
# !pip install pandas numpy scikit-learn matplotlib seaborn openpyxl -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # non-interactive backend, safe for Colab
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate, KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---- Upload the Excel file ----
from google.colab import files
print('Please upload BetasCorrelation.xlsx')
uploaded = files.upload()
EXCEL_FILE = list(uploaded.keys())[0]  # picks whatever file was uploaded
print(f'\nFile received: {EXCEL_FILE}')


## Step 1 — Load Data and Auto-Identify Columns

In [ ]:
# Read Sheet 1 only (index 0); Sheet 2 is not used for modeling
df = pd.read_excel(EXCEL_FILE, sheet_name=0)

# --- Auto-identify column roles ---
# Column A  → subject IDs (first column)
# Columns B–I (positions 1–8) → 8 beta outcome variables
# Columns J–AC (positions 9–29) → 21 questionnaire predictor variables
subject_col    = df.columns[0]
beta_cols      = df.columns[1:9].tolist()
predictor_cols = df.columns[9:].tolist()

print('=' * 60)
print('DATASET SUMMARY')
print('=' * 60)
print(f'  Number of subjects           : {len(df)}')
print(f'  Number of beta outcomes      : {len(beta_cols)}')
print(f'  Number of questionnaire preds: {len(predictor_cols)}')
print(f'\n  Subject ID column : {subject_col}')
print(f'\n  Beta outcome columns ({len(beta_cols)}):')
for c in beta_cols:
    print(f'    {c}')
print(f'\n  Questionnaire predictor columns ({len(predictor_cols)}):')
for c in predictor_cols:
    print(f'    {c}')

print('\n--- Descriptive Statistics: Beta Values ---')
print(df[beta_cols].describe().round(4))
print('\n--- Descriptive Statistics: Questionnaire Variables ---')
print(df[predictor_cols].describe().round(4))
print(f'\nMissing values in dataset: {df.isnull().sum().sum()}')


## Step 2 — Exploratory Correlation Heatmap

In [ ]:
# Correlations between each beta outcome and each questionnaire variable
corr_matrix = df[beta_cols + predictor_cols].corr().loc[beta_cols, predictor_cols]

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.4, annot_kws={'size': 7},
    ax=ax
)
ax.set_title('Pearson Correlations: Beta Values vs. Questionnaire Variables', fontsize=13, pad=12)
ax.set_xlabel('Questionnaire Predictors', fontsize=10)
ax.set_ylabel('Brain-Region Beta Outcomes', fontsize=10)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=9)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: correlation_heatmap.png')


## Step 3 — Define Regression Pipelines

In [ ]:
# Feature matrix (predictors) — same for all beta outcomes
X = df[predictor_cols].values

# Alpha search grids for regularized models (log-spaced, reasonable for N=81)
alphas = np.logspace(-3, 3, 50)

def make_pipelines():
    """
    Returns a dict of three scikit-learn pipelines.
    StandardScaler is INSIDE the pipeline so it fits only on training data
    during cross-validation — no data leakage.
    """
    return {
        'LinearRegression': Pipeline([
            ('scaler', StandardScaler()),
            ('model',  LinearRegression())
        ]),
        'Ridge': Pipeline([
            ('scaler', StandardScaler()),
            # RidgeCV tunes alpha via internal leave-one-out CV on the training fold
            ('model',  RidgeCV(alphas=alphas, cv=None))
        ]),
        'Lasso': Pipeline([
            ('scaler', StandardScaler()),
            # LassoCV tunes alpha via 5-fold CV on the training fold
            ('model',  LassoCV(alphas=alphas, cv=5, max_iter=10000, random_state=42))
        ]),
    }

def cv_metrics(pipeline, X, y, cv):
    """Run cross-validation and return mean ± std of R², RMSE, MAE."""
    scoring = {
        'r2':   'r2',
        'nmse': 'neg_mean_squared_error',
        'nmae': 'neg_mean_absolute_error',
    }
    results = cross_validate(pipeline, X, y, cv=cv, scoring=scoring,
                              return_train_score=False, n_jobs=-1)
    r2_vals   = results['test_r2']
    rmse_vals = np.sqrt(-results['test_nmse'])
    mae_vals  = -results['test_nmae']
    return {
        'R2_mean':   r2_vals.mean(),   'R2_std':   r2_vals.std(),
        'RMSE_mean': rmse_vals.mean(), 'RMSE_std': rmse_vals.std(),
        'MAE_mean':  mae_vals.mean(),  'MAE_std':  mae_vals.std(),
    }

def train_test_metrics(pipeline, X, y, test_size=0.2, random_state=42):
    """Single 80/20 train-test split evaluation."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    return {
        'R2':   r2_score(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE':  mean_absolute_error(y_test, y_pred),
        'y_test': y_test,
        'y_pred': y_pred,
    }

print('Pipelines defined. Ready to run regressions.')


## Step 4 — Run All Regression Pipelines

One pipeline per beta outcome × three models × three evaluation strategies.

In [ ]:
cv5  = KFold(n_splits=5,  shuffle=True, random_state=42)
cv10 = KFold(n_splits=10, shuffle=True, random_state=42)

# Storage for results
records_5fold  = []
records_10fold = []
records_tt     = []   # train-test split
lasso_features = []   # selected features per beta
best_model_summary = []
tt_predictions = {}   # {beta_col: {model_name: (y_test, y_pred)}}

print('Running regressions... (this may take ~30 seconds)\n')

for beta in beta_cols:
    y = df[beta].values
    tt_predictions[beta] = {}
    print(f'  Processing: {beta}')

    model_r2_tt = {}  # to pick best model for this beta

    for model_name, pipeline in make_pipelines().items():

        # --- 5-fold CV ---
        m5 = cv_metrics(pipeline, X, y, cv5)
        records_5fold.append({'Beta': beta, 'Model': model_name, **m5})

        # --- 10-fold CV ---
        m10 = cv_metrics(pipeline, X, y, cv10)
        records_10fold.append({'Beta': beta, 'Model': model_name, **m10})

        # --- 80/20 Train-Test Split ---
        tt = train_test_metrics(pipeline, X, y)
        records_tt.append({
            'Beta':  beta, 'Model': model_name,
            'R2': tt['R2'], 'RMSE': tt['RMSE'], 'MAE': tt['MAE'],
        })
        tt_predictions[beta][model_name] = (tt['y_test'], tt['y_pred'])
        model_r2_tt[model_name] = tt['R2']

        # --- Lasso: record selected features (refit on full data for reporting) ---
        if model_name == 'Lasso':
            pipeline.fit(X, y)
            lasso_model = pipeline.named_steps['model']
            coefs = lasso_model.coef_
            best_alpha = lasso_model.alpha_
            for feat, coef in zip(predictor_cols, coefs):
                if coef != 0:
                    lasso_features.append({
                        'Beta': beta, 'Feature': feat,
                        'Coefficient': round(coef, 6),
                        'BestAlpha': round(best_alpha, 6),
                    })

    # Best model for this beta (highest R² on train-test split)
    best_model = max(model_r2_tt, key=model_r2_tt.get)
    best_model_summary.append({
        'Beta': beta, 'BestModel': best_model,
        'BestR2_TT': round(model_r2_tt[best_model], 4),
    })

print('\nDone! All regressions complete.')


## Step 5 — Save Performance CSVs

In [ ]:
df_5fold  = pd.DataFrame(records_5fold).round(4)
df_10fold = pd.DataFrame(records_10fold).round(4)
df_tt     = pd.DataFrame(records_tt).round(4)
df_lasso  = pd.DataFrame(lasso_features)
df_best   = pd.DataFrame(best_model_summary)

df_5fold.to_csv('model_performance_5fold.csv',  index=False)
df_10fold.to_csv('model_performance_10fold.csv', index=False)
df_tt.to_csv('train_test_performance.csv',       index=False)
df_lasso.to_csv('lasso_selected_features.csv',   index=False)
df_best.to_csv('best_model_summary.csv',         index=False)

print('Saved CSVs:')
for f in ['model_performance_5fold.csv', 'model_performance_10fold.csv',
          'train_test_performance.csv',   'lasso_selected_features.csv',
          'best_model_summary.csv']:
    print(f'  {f}')

print('\n--- 5-Fold CV Results ---')
print(df_5fold.to_string(index=False))
print('\n--- Train-Test Split Results ---')
print(df_tt.to_string(index=False))


## Step 6 — Model Performance Comparison Chart

In [ ]:
models  = df_5fold['Model'].unique()
betas   = df_5fold['Beta'].unique()
x       = np.arange(len(betas))
width   = 0.25
colors  = ['#4C72B0', '#DD8452', '#55A868']

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=False)
metric_pairs = [
    ('R2_mean',   'R2_std',   'R² (5-Fold CV)'),
    ('RMSE_mean', 'RMSE_std', 'RMSE (5-Fold CV)'),
    ('MAE_mean',  'MAE_std',  'MAE (5-Fold CV)'),
]

for ax, (mean_col, std_col, ylabel) in zip(axes, metric_pairs):
    for i, (model, color) in enumerate(zip(models, colors)):
        subset = df_5fold[df_5fold['Model'] == model]
        ax.bar(x + i * width, subset[mean_col].values, width,
               yerr=subset[std_col].values,
               label=model, color=color, capsize=3, alpha=0.85)
    ax.set_title(ylabel, fontsize=11)
    ax.set_xticks(x + width)
    ax.set_xticklabels(betas, rotation=45, ha='right', fontsize=8)
    ax.set_xlabel('Brain Region (Beta)', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('Model Performance Comparison — 5-Fold Cross-Validation\n(error bars = ±1 SD across folds)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('model_performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_performance_comparison.png')


## Step 7 — Predicted vs. Actual Plots (Best Model per Beta)

In [ ]:
n_betas = len(beta_cols)
ncols   = 4
nrows   = int(np.ceil(n_betas / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4))
axes = axes.flatten()

for i, beta in enumerate(beta_cols):
    ax = axes[i]
    best_model_name = df_best.loc[df_best['Beta'] == beta, 'BestModel'].values[0]
    y_test, y_pred  = tt_predictions[beta][best_model_name]

    ax.scatter(y_test, y_pred, alpha=0.75, edgecolors='k', linewidths=0.4,
               s=50, color='#4C72B0')

    # Perfect prediction reference line
    lims = [min(y_test.min(), y_pred.min()) - 0.1,
            max(y_test.max(), y_pred.max()) + 0.1]
    ax.plot(lims, lims, 'r--', linewidth=1.2, label='Ideal')

    r2 = r2_score(y_test, y_pred)
    ax.set_title(f'{beta}\nBest: {best_model_name}  R²={r2:.3f}', fontsize=9)
    ax.set_xlabel('Actual', fontsize=8)
    ax.set_ylabel('Predicted', fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, linestyle='--', alpha=0.4)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Predicted vs. Actual — Best Model per Beta Outcome (80/20 Test Split)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: predicted_vs_actual.png')


## Step 8 — Lasso Coefficient Plots

In [ ]:
if df_lasso.empty:
    print('Lasso selected zero features across all beta outcomes.')
    print('This is common with small N. Ridge may be more appropriate.')
else:
    n_betas_with_features = df_lasso['Beta'].nunique()
    ncols = min(4, n_betas_with_features)
    nrows = int(np.ceil(n_betas_with_features / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4))
    axes = np.array(axes).flatten() if n_betas_with_features > 1 else [axes]

    plot_idx = 0
    for beta in beta_cols:
        subset = df_lasso[df_lasso['Beta'] == beta].sort_values('Coefficient')
        if subset.empty:
            continue
        ax = axes[plot_idx]
        bar_colors = ['#DD8452' if c > 0 else '#4C72B0' for c in subset['Coefficient']]
        ax.barh(subset['Feature'], subset['Coefficient'],
                color=bar_colors, edgecolor='k', linewidth=0.4)
        ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
        ax.set_title(f'Lasso Coefficients: {beta}\n(alpha={subset["BestAlpha"].iloc[0]})', fontsize=9)
        ax.set_xlabel('Coefficient Value', fontsize=8)
        ax.tick_params(axis='y', labelsize=7)
        ax.grid(axis='x', linestyle='--', alpha=0.4)
        plot_idx += 1

    for j in range(plot_idx, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Lasso Selected Features per Brain Region\n'
                 '(orange = positive effect, blue = negative effect)',
                 fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('lasso_coefficients.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: lasso_coefficients.png')


## Final Summary

In [ ]:
print('=' * 65)
print('FINAL SUMMARY')
print('=' * 65)

print('\n1. BEST MODEL PER BETA OUTCOME (by R² on 80/20 test split):\n')
print(df_best.to_string(index=False))

print('\n2. AVERAGE PERFORMANCE ACROSS ALL BETA OUTCOMES (5-Fold CV):\n')
avg_perf = df_5fold.groupby('Model')[['R2_mean', 'RMSE_mean', 'MAE_mean']].mean().round(4)
print(avg_perf.to_string())

print('\n3. MOST FREQUENTLY SELECTED LASSO FEATURES (across all beta outcomes):\n')
if df_lasso.empty:
    print('  Lasso selected no features (all coefficients shrunk to zero).')
    print('  This is common with very small N. Ridge may be more appropriate here.')
else:
    freq = (df_lasso.groupby('Feature')['Beta']
            .count()
            .rename('NumBetasSelected')
            .sort_values(ascending=False))
    print(freq.to_string())

print('\n' + '=' * 65)
print('OUTPUT FILES:')
for f in ['model_performance_5fold.csv', 'model_performance_10fold.csv',
          'train_test_performance.csv',   'lasso_selected_features.csv',
          'best_model_summary.csv',       'correlation_heatmap.png',
          'model_performance_comparison.png', 'predicted_vs_actual.png',
          'lasso_coefficients.png']:
    print(f'  {f}')
print('=' * 65)
print('\nAll done! Download files from the Files panel on the left.')
